In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number
import random


In [0]:
# Number of rows
num_rows = 1000000   # change to 5000000 if needed

# Base DataFrame
df = spark.range(0, num_rows)

regions = ["East", "West", "North", "South"]
categories = ["Electronics", "Furniture", "Clothing"]
products = {
    "Electronics": ["Laptop", "Phone", "Tablet", "Mouse", "Monitor"],
    "Furniture": ["Chair", "Desk", "Sofa", "Bed"],
    "Clothing": ["Shirt", "Jeans", "Jacket", "Shoes"]
}

sales_people = [f"Sales_{i}" for i in range(1, 51)]

# Create large dataset
sales_df = (
    df
    .withColumn("order_id", concat(lit("ID_"), col("id") + 1))
    .withColumn("order_date",
                expr("date_add('2023-01-01', cast(rand()*730 as int))"))
    .withColumn("region",
                element_at(array(*[lit(x) for x in regions]),
                           (rand()*4 + 1).cast("int")))
    .withColumn("category",
                element_at(array(*[lit(x) for x in categories]),
                           (rand()*3 + 1).cast("int")))
    .withColumn("sales_person",
                element_at(array(*[lit(x) for x in sales_people]),
                           (rand()*50 + 1).cast("int")))
    .withColumn("sales_amount",
                round(rand()*100000 + 1000, 2))
    .drop("id")
)

In [0]:
sales_df.show(5)

+--------+----------+------+-----------+------------+------------+
|order_id|order_date|region|   category|sales_person|sales_amount|
+--------+----------+------+-----------+------------+------------+
|    ID_1|2023-03-26| North|  Furniture|    Sales_34|    24577.91|
|    ID_2|2024-04-25|  West|   Clothing|    Sales_19|    80280.23|
|    ID_3|2024-10-03|  East|  Furniture|    Sales_36|    78772.75|
|    ID_4|2023-10-30| South|   Clothing|    Sales_40|    10185.53|
|    ID_5|2024-08-25|  West|Electronics|    Sales_48|    27606.78|
+--------+----------+------+-----------+------------+------------+
only showing top 5 rows


In [0]:
sales_df.createOrReplaceTempView("sales_df")

In [0]:
# Question 1: Rank Sales Within Each Region
result = spark.sql("""
SELECT
order_id,region,sales_amount ,row_number() over(Partition by region order by sales_amount Desc) as Rank 
from sales_df 
""")

display(result)

order_id,region,sales_amount,Rank
ID_524907,South,100999.63,1
ID_637070,South,100999.53,2
ID_867161,South,100999.07,3
ID_597889,South,100998.93,4
ID_918727,South,100998.93,5
ID_174020,South,100998.71,6
ID_288069,South,100998.63,7
ID_69599,South,100998.26,8
ID_82599,South,100997.67,9
ID_564545,South,100997.67,10


In [0]:
# SQL syntax
# Question 2: Top 5 Sales Per Region
result = spark.sql("""
with rank_sales as (
    select order_id,region,sales_amount, row_number() over(partition by region order by sales_amount Desc)
    as rank from sales_df )
select * from rank_sales where rank <= 5
""")
display(result)

order_id,region,sales_amount,rank
ID_870238,East,100999.14,1
ID_969397,East,100999.14,2
ID_728520,East,100998.99,3
ID_763471,East,100998.57,4
ID_295582,East,100998.54,5
ID_613874,North,100999.43,1
ID_892458,North,100999.09,2
ID_321548,North,100998.84,3
ID_822667,North,100998.19,4
ID_616881,North,100997.73,5


In [0]:
# pyspark syntax
window_rank = Window.partitionBy(col("region")).orderBy(col("sales_amount").desc())
result = sales_df.withColumn("REGION_RANK", row_number().over(window_rank))\
         .filter(col('REGION_RANK') <=5)\
         .select(col("order_id"),col("region"),col("sales_amount"),col("REGION_RANK"))
display(result)

order_id,region,sales_amount,REGION_RANK
ID_870238,East,100999.14,1
ID_969397,East,100999.14,2
ID_728520,East,100998.99,3
ID_763471,East,100998.57,4
ID_295582,East,100998.54,5
ID_613874,North,100999.43,1
ID_892458,North,100999.09,2
ID_321548,North,100998.84,3
ID_822667,North,100998.19,4
ID_616881,North,100997.73,5


In [0]:
# Question 3: Running Total of Sales Per Region (By Date)
result_df = spark.sql("""
        SELECT
    order_date,region,sales_amount,
    SUM(sales_amount) OVER (PARTITION BY region ORDER BY order_date
    ) AS running_total
FROM sales_df
""")
display(result_df)

order_date,region,sales_amount,running_total
2023-01-01,South,21075.52,1.7620128309999995E7
2023-01-01,South,34209.25,1.7620128309999995E7
2023-01-01,South,21322.6,1.7620128309999995E7
2023-01-01,South,34951.9,1.7620128309999995E7
2023-01-01,South,18949.23,1.7620128309999995E7
2023-01-01,South,37888.3,1.7620128309999995E7
2023-01-01,South,28462.11,1.7620128309999995E7
2023-01-01,South,36728.03,1.7620128309999995E7
2023-01-01,South,39416.86,1.7620128309999995E7
2023-01-01,South,64010.17,1.7620128309999995E7


In [0]:
window_region = Window.partitionBy(col('region')).orderBy(col('Order_date')).rowsBetween(Window.unboundedPreceding, Window.currentRow)
result_df = sales_df.withColumn('running_total',sum(col('sales_amount')).over(window_region))\
    .select(col('order_date'),col('region'),col('sales_amount'),col('running_total'))
display(result_df)

order_date,region,sales_amount,running_total
2023-01-01,South,21075.52,21075.52
2023-01-01,South,34209.25,55284.770000000004
2023-01-01,South,21322.6,76607.37
2023-01-01,South,34951.9,111559.26999999999
2023-01-01,South,18949.23,130508.49999999999
2023-01-01,South,37888.3,168396.8
2023-01-01,South,28462.11,196858.90999999997
2023-01-01,South,36728.03,233586.93999999997
2023-01-01,South,39416.86,273003.8
2023-01-01,South,64010.17,337013.97


In [0]:
# Question 4: Previous Sale Per Sales Person
previous_sale =  spark.sql("""
   SELECT
    sales_person,
    order_date,
    sales_amount,
    LAG(sales_amount, 1) OVER (
        PARTITION BY sales_person
        ORDER BY order_date
    ) AS previous_sale
FROM sales_df
ORDER BY sales_person, order_date;                        
""")
previous_sale.display()

sales_person,order_date,sales_amount,previous_sale
Sales_1,2023-01-01,100807.06,28998.97
Sales_1,2023-01-01,51095.75,61096.81
Sales_1,2023-01-01,50340.55,70472.3
Sales_1,2023-01-01,89445.83,9287.6
Sales_1,2023-01-01,41954.07,55305.52
Sales_1,2023-01-01,58545.15,44413.97
Sales_1,2023-01-01,32804.33,41954.07
Sales_1,2023-01-01,70472.3,23887.97
Sales_1,2023-01-01,28998.97,43843.17
Sales_1,2023-01-01,89953.87,59300.09


In [0]:
sale_window = Window.partitionBy(col("sales_person")).orderBy(col("order_date"))
previous_sale = sales_df.withColumn("previous_sale", lag(col("sales_amount"), 1).over(sale_window))\
    .select(col("sales_person"),col("order_date"),col("sales_amount"),col("previous_sale"))
display(previous_sale)

sales_person,order_date,sales_amount,previous_sale
Sales_11,2023-01-01,70893.08,null
Sales_11,2023-01-01,40750.66,70893.08
Sales_11,2023-01-01,88629.95,40750.66
Sales_11,2023-01-01,78128.42,88629.95
Sales_11,2023-01-01,21736.01,78128.42
Sales_11,2023-01-01,7724.9,21736.01
Sales_11,2023-01-01,31563.46,7724.9
Sales_11,2023-01-01,48227.22,31563.46
Sales_11,2023-01-01,72288.2,48227.22
Sales_11,2023-01-01,13753.21,72288.2


In [0]:
# Question 5: Difference from Previous Sale

In [0]:
# Question 6: Top 3 Sales Person Per Region Based on Total Sales
# Question 7: Month-over-Month Growth Per Region
# Question 8: Percentage Contribution to Region Total
# Question 9: Moving Average (Last 5 Sales Per Region)

In [0]:
sales_df.show(5)

+--------+----------+------+-----------+------------+------------+
|order_id|order_date|region|   category|sales_person|sales_amount|
+--------+----------+------+-----------+------------+------------+
|    ID_1|2023-03-26| North|  Furniture|    Sales_34|    24577.91|
|    ID_2|2024-04-25|  West|   Clothing|    Sales_19|    80280.23|
|    ID_3|2024-10-03|  East|  Furniture|    Sales_36|    78772.75|
|    ID_4|2023-10-30| South|   Clothing|    Sales_40|    10185.53|
|    ID_5|2024-08-25|  West|Electronics|    Sales_48|    27606.78|
+--------+----------+------+-----------+------------+------------+
only showing top 5 rows


In [0]:
# Question 6: Top 3 Sales Person Per Region Based on Total Sales
result = spark.sql("""
     with sales_rank as(
         Select sales_person,region,Round(sum(sales_amount),2) as total_sales,row_number() over(partition by region order by
         sum(sales_amount) desc) as rank  from sales_df group by sales_person,region)
         select sales_person,region,total_sales,rank from sales_rank where rank <=3        
""")
result.display()

sales_person,region,total_sales,rank
Sales_6,East,2.6371995907E8,1
Sales_11,East,2.6286434629E8,2
Sales_49,East,2.6258635668E8,3
Sales_31,North,2.6324142192E8,1
Sales_16,North,2.628842178E8,2
Sales_24,North,2.6257420171E8,3
Sales_40,South,2.6294016826E8,1
Sales_15,South,2.6170760821E8,2
Sales_24,South,2.6143787978E8,3
Sales_48,West,2.657953589E8,1


In [0]:
# Question 6: Top 3 Sales Person Per Region Based on Total Sales
sales_total = sales_df.groupBy("sales_person","region") \
                      .agg(round(sum("sales_amount"),2).alias("total_sales"))

window_spec = Window.partitionBy("region").orderBy(sales_total["total_sales"].desc())

result = sales_total.withColumn("rank", row_number().over(window_spec)) \
                    .filter("rank <= 3")

result.show()

+------------+------+--------------+----+
|sales_person|region|   total_sales|rank|
+------------+------+--------------+----+
|     Sales_6|  East|2.6371995907E8|   1|
|    Sales_11|  East|2.6286434629E8|   2|
|    Sales_49|  East|2.6258635668E8|   3|
|    Sales_31| North|2.6324142192E8|   1|
|    Sales_16| North| 2.628842178E8|   2|
|    Sales_24| North|2.6257420171E8|   3|
|    Sales_40| South|2.6294016826E8|   1|
|    Sales_15| South|2.6170760821E8|   2|
|    Sales_24| South|2.6143787978E8|   3|
|    Sales_48|  West| 2.657953589E8|   1|
|    Sales_35|  West|2.6503347537E8|   2|
|    Sales_14|  West|2.6198808057E8|   3|
+------------+------+--------------+----+

